# Budget Auction — Experiment 2: Multi-Good Privately Correlated Budget Auction

Compares variants **A** / **B** / **C** in a setting *beyond* Deng 2021: types are multi-dimensional and transitions are privately correlated. The key hypothesis: richer history compression should outperform GBAM-style compressed state here.

In [ ]:
import os
# 1. Force remove existing JAX/PJRT installations to ensure a clean state
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt

# 2. Install the latest JAX with full CUDA 12 support and updated libraries
!pip install -U "jax[cuda12]" dm-haiku optax flax

print('\n--- UPGRADE FINISHED ---')
print('IMPORTANT: Please go to Runtime -> Restart session now to activate the GPU drivers.')


In [ ]:
# Mount Google Drive after restarting the session
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/My\ Research/Dynamic\ Mechanism\ Design/Budget\ Auction
except ImportError:
    pass


In [ ]:
from typing import Dict
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.private_budget_multigood_env import create_private_budget_multigood_auction
from budget_auction.baselines.static_myopic import run_static_myopic_baseline
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report

print('JAX devices:', jax.devices())


## Configuration

In [ ]:
VARIANT = 'A'   # 'A' | 'B' | 'C'
SEED    = 42

cfg = dict(
    n_buyers=2, n_goods=2, horizon=5, discount=0.9,
    initial_budget=3.0,
    profit_goal=2.0,         # default; Stage 2 search may revise this
    num_outer_epochs=2000,
    num_inner_steps=600,     # 600 steps for budget IC (multi-stage oracle)
    lr_theta=1e-4, lr_w=3e-4,
    num_samples_train=128,   # 128 samples to reduce regret-estimate variance
    alpha_weight=1e5, beta_weight=1e3,
    print_iter=100,
    policy_hidden=128, residual_init_std=1.0,
)
print('Config:', cfg)


## Build Auction Environment

In [ ]:
auction = create_private_budget_multigood_auction(
    n_buyers=cfg['n_buyers'], n_goods=cfg['n_goods'],
    horizon=cfg['horizon'], discount=cfg['discount'],
    initial_budget=cfg['initial_budget'], seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')
print(f'Type space: [{auction.types_low}, {auction.types_high}]')


In [ ]:
key = jax.random.PRNGKey(0)
s0 = auction.get_initial_state(key)
print(f'types={s0.current_type}  alpha={s0.alpha}  beta={s0.beta}')


## Check Type Transition Dynamics

In [ ]:
from budget_auction.envs.private_budget_multigood_env import private_budget_transition_fn
alloc_hist = jnp.zeros((auction.num_horizons, auction.num_buyers, auction.num_goods))
keys_t = jax.random.split(jax.random.PRNGKey(1), 20)
samples = jnp.stack([auction.transition_function(auction, 0, s0.current_type, alloc_hist, k)
                     for k in keys_t])
print(f'Current types: {s0.current_type}')
print(f'Next types mean: {jnp.mean(samples, 0)}')
print(f'Next types std:  {jnp.std(samples, 0)}')
print('(Should be near current types; sigma_no=0.01 → small drift without allocation)')


## Evaluate Myopic Baseline

In [ ]:
print('--- Baseline B1: Static Myopic ---')
b1 = run_static_myopic_baseline(auction, num_samples=256, num_keys=16, base_seed=1)
for k, v in b1.items():
    if isinstance(v, float): print(f'  {k}: {v:.4f}')
print('\n(Use baseline revenue to calibrate profit_goal search range above.)')


## Quick Single-Variant Training

Sanity check with default hyperparameters. Skip to **Hyperparameter Search** for the full pipeline.

In [ ]:
_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
_pnet = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, G=auction.num_goods,
    hidden=cfg['policy_hidden'], residual_init_std=cfg['residual_init_std'],
)
logs = nested_GD_budget(
    auction=auction, mechanism_network=_mnet, policy_network=_pnet,
    profit_goal=cfg['profit_goal'], num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=cfg['lr_theta'], num_inner_steps=cfg['num_inner_steps'],
    lr_w=cfg['lr_w'], num_samples_train=cfg['num_samples_train'],
    alpha_weight=cfg['alpha_weight'], beta_weight=cfg['beta_weight'],
    print_iter=cfg['print_iter'], seed=SEED,
)
mechanism_params = logs.pop('mechanism_params')
policy_params    = logs.pop('policy_params')
print('Quick training complete.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(logs['reg_plus']); axes[0].set_title('Regret+ ' + VARIANT)
axes[1].plot(logs['gap_plus']); axes[1].set_title('Profit Gap+')
axes[2].plot(logs['ir']);        axes[2].set_title('IR Loss')
for _ax in axes: _ax.set_xlabel('Epoch')
plt.suptitle(f'Exp 2 — Quick check, Variant {VARIANT} ({auction.num_goods} goods)', y=1.02)
plt.tight_layout(); plt.show()


## Hyperparameter Search — 3-Stage Staged Pipeline

Hyperparameters are tuned in three sequential stages, each building on the previous, so that only the parameters relevant to each stage are free while the others are held fixed. Results are written to disk after every trial; re-running the cell resumes from where it left off.

| Stage | Free parameters | Fixed | Trials |
|-------|----------------|-------|--------|
| **1** | `lr_theta`, `lr_w` | moderate weights, default profit_goal | 12 |
| **2** | `profit_goal`, `alpha_weight`, `beta_weight` | lr* from S1 | 36 |
| **3** | `num_inner_steps` | all S1+S2 best values | 4 |

**Scoring** (all stages): `revenue − 10·max(0, IC−ε) − 10·max(0, IR−ε) − 10·max(0, IR_viol−ε) − 10·max(0, BV−ε)`  
Revenue is the objective; IC, IR, and feasibility are explicit constraints.

In [ ]:
import itertools, random, time, math

# ---- Global time budget ----
HP_TIME_BUDGET_HOURS = 12

# ---- Per-stage epoch budgets ----
HP_S1_EPOCHS = 300   # Stage 1: LR anchor  (fast, just need stability signal)
HP_S2_EPOCHS = 500   # Stage 2: weights + profit_goal
HP_S3_EPOCHS = 500   # Stage 3: inner steps
HP_SAMPLES   = 64    # MC samples during training (all stages)
HP_EVAL_KEYS = 16    # eval keys per trial
HP_EVAL_SAMPS = 128  # eval MC samples per trial

# ---- Constraint tolerances ----
# IC and IR are *constraints*, not objective terms.
# Revenue is maximised freely; violations are penalised once they
# exceed tolerance (10 units penalty per unit of excess).
HP_IC_LOSS_THRESH = 0.05   # cumulative regret
HP_IR_LOSS_THRESH = 0.05   # sum |reward|
HP_IR_VIOL_THRESH = 0.05   # IR violation rate
HP_BV_THRESH      = 0.05   # budget violation rate

hp_save_dir = 'outputs/private_budget_multigood/hparam_search'
os.makedirs(hp_save_dir, exist_ok=True)

# ---- Scoring function ----
def _hp_score(r):
    bv_pen = 10.0 * max(0.0, r['budget_viol_mean'] - HP_BV_THRESH)
    iv_pen = 10.0 * max(0.0, r['ir_viol_mean']    - HP_IR_VIOL_THRESH)
    ir_pen = 10.0 * max(0.0, r['ir_loss_mean']    - HP_IR_LOSS_THRESH)
    _rg = r.get('regret_mean', float('nan'))
    ic_pen = 0.0 if (_rg != _rg) else 10.0 * max(0.0, _rg - HP_IC_LOSS_THRESH)
    return r['revenue_mean'] - bv_pen - iv_pen - ir_pen - ic_pen

# ---- Single-trial runner ----
# trial_cfg must contain all keys needed by nested_GD_budget.
def _run_trial(trial_cfg, trial_seed, n_epochs, n_samples):
    _ni = trial_cfg['num_inner_steps']
    _pg = trial_cfg['profit_goal']
    _mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
    _pnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=cfg['policy_hidden'],
        residual_init_std=cfg['residual_init_std'],
    )
    _tl = nested_GD_budget(
        auction=auction,
        mechanism_network=_mnet, policy_network=_pnet,
        profit_goal=_pg,
        num_outer_epochs=n_epochs,
        lr_theta=trial_cfg['lr_theta'],
        num_inner_steps=_ni,
        lr_w=trial_cfg['lr_w'],
        num_samples_train=n_samples,
        alpha_weight=trial_cfg['alpha_weight'],
        beta_weight=trial_cfg['beta_weight'],
        print_iter=10**7,
        seed=trial_seed,
    )
    _mp = _tl.pop('mechanism_params'); _pp = _tl.pop('policy_params')
    _ek = jax.random.split(jax.random.PRNGKey(trial_seed + 99999), HP_EVAL_KEYS)
    _ev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_mnet, mechanism_params=_mp,
        profit_goal=_pg, num_samples=HP_EVAL_SAMPS, keys=_ek,
        policy_network=_pnet, policy_params=_pp,
    )
    return {
        'config':           trial_cfg,
        'revenue_mean':     float(_ev['profit_stats']['mean']),
        'revenue_std':      float(_ev['profit_stats']['std']),
        'ir_loss_mean':     float(_ev['ir_loss_stats']['mean']),
        'budget_viol_mean': float(_ev['budget_viol_stats']['mean']),
        'ir_viol_mean':     float(_ev['ir_viol_stats']['mean']),
        'gap_plus_mean':    float(_ev['gap_plus_stats']['mean']),
        'regret_mean':      float(_ev.get('regret_stats', {}).get('mean', float('nan'))),
        'final_reg_train':  float(_tl['reg'][-1]) if _tl.get('reg') else float('nan'),
        'final_ir_train':   float(_tl['ir'][-1])  if _tl.get('ir')  else float('nan'),
        'seed':             trial_seed,
    }

# ---- Stage runner (handles resumption, time budget, fault tolerance) ----
def _run_stage(stage_name, combos, fixed_cfg, n_epochs, n_samples,
               results_path, search_start, budget_s):
    if os.path.exists(results_path):
        with open(results_path) as _f:
            _results = json.load(_f)
        print(f'  [{stage_name}] loaded {len(_results)} prior results (resuming)')
        _done = {str(r['config']) for r in _results}
    else:
        _results = []; _done = set()
    for _i, _tcfg in enumerate(combos):
        _full = {**fixed_cfg, **_tcfg}   # search overrides fixed
        if str(_full) in _done:
            continue
        _el = time.time() - search_start
        if _el >= budget_s:
            print(f'  [{stage_name}] time budget reached at trial {_i}'); break
        _rem = (budget_s - _el) / 3600
        print(f'  [{stage_name}] {_i+1}/{len(combos)} rem={_rem:.2f}h | {_tcfg}')
        _tseed = SEED + abs(hash(stage_name)) % 9999 + _i
        try:
            _rec = _run_trial(_full, _tseed, n_epochs, n_samples)
            _rec['score'] = _hp_score(_rec)
            _results.append(_rec); _done.add(str(_full))
            with open(results_path, 'w') as _f:
                json.dump(_results, _f, indent=2)
            _rv = _rec['revenue_mean']; _sc = _rec['score']
            _rg = _rec.get('regret_mean', float('nan'))
            _ir = _rec['ir_loss_mean']
            print(f'    => rev={_rv:.4f} regret={_rg:.4f} ir={_ir:.4f} score={_sc:.4f}')
        except Exception as _exc:
            print(f'    [FAILED] {_exc}')
    return sorted(_results, key=lambda r: r['score'], reverse=True)

# Shared timer (covers all three stages)
_search_start = time.time()
_budget_s     = HP_TIME_BUDGET_HOURS * 3600
print(f'Search started. Budget: {HP_TIME_BUDGET_HOURS}h')
print(f'Thresholds — IC<={HP_IC_LOSS_THRESH} IR<={HP_IR_LOSS_THRESH} '
      f'IV<={HP_IR_VIOL_THRESH} BV<={HP_BV_THRESH}')


### Stage 1 — Anchor Learning Rates

Adam's adaptive gradient scaling makes `lr_theta` and `lr_w` partially independent of loss weight magnitudes, so they can be anchored before the weights are tuned. We hold `alpha_weight`, `beta_weight`, and `profit_goal` at moderate defaults and search a 4×3 = 12-trial grid.

Criterion: score (revenue subject to IC/IR constraints). A large spread in scores here indicates strong LR sensitivity; a flat landscape means any LR in the range is acceptable.

In [ ]:
_s1_search = {
    'lr_theta': [3e-5, 1e-4, 3e-4, 1e-3],
    'lr_w':     [1e-4, 3e-4, 1e-3],
}
_s1_fixed = {
    'alpha_weight':    1e5,
    'beta_weight':     1e3,
    'profit_goal':     cfg['profit_goal'],
    'num_inner_steps': 150,
}
_s1_combos = [dict(zip(_s1_search, c))
              for c in itertools.product(*_s1_search.values())]
random.seed(SEED); random.shuffle(_s1_combos)
print(f'Stage 1: {len(_s1_combos)} combos x {HP_S1_EPOCHS} epochs')

_s1_results = _run_stage(
    'S1-LR', _s1_combos, _s1_fixed, HP_S1_EPOCHS, HP_SAMPLES,
    os.path.join(hp_save_dir, 's1_lr.json'), _search_start, _budget_s,
)

lr_theta_star = _s1_results[0]['config']['lr_theta']
lr_w_star     = _s1_results[0]['config']['lr_w']
print(f'\nStage 1 best: lr_theta={lr_theta_star:.1e}  lr_w={lr_w_star:.1e}')
print(f'  score={_s1_results[0]["score"]:.4f}  revenue={_s1_results[0]["revenue_mean"]:.4f}')

# Sensitivity plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for _ax, _key, _star in zip(axes, ['lr_theta', 'lr_w'], [lr_theta_star, lr_w_star]):
    _vals = sorted(set(r['config'][_key] for r in _s1_results))
    _means = []
    for _v in _vals:
        _sub = [r['score'] for r in _s1_results if r['config'][_key] == _v]
        _ax.scatter([_v]*len(_sub), _sub, alpha=0.6, s=40)
        _means.append(sum(_sub)/len(_sub))
    _ax.plot(_vals, _means, 'k--', alpha=0.4, label='mean')
    _ax.axvline(_star, color='red', linestyle='--', label=f'best={_star:.0e}')
    _ax.set_xscale('log'); _ax.set_xlabel(_key)
    _ax.set_ylabel('Score'); _ax.legend(); _ax.set_title(f'Stage 1 — {_key}')
plt.tight_layout(); plt.show()


### Stage 2 — Joint Search: Profit Goal × Loss Weights

`profit_goal`, `alpha_weight`, and `beta_weight` are tightly coupled: raising `profit_goal` increases the gap-term magnitude, making the IC penalty relatively weaker unless `alpha_weight` is raised proportionally. Searching them jointly (with LRs anchored) reveals the correct trade-off surface and identifies the highest revenue target that still converges to a feasible mechanism.

**Fixed:** `lr_theta*`, `lr_w*` from Stage 1, `num_inner_steps=200`  
**Search:** `profit_goal` × `alpha_weight` × `beta_weight` (3×3×3 = **27 trials**)

In [ ]:
_s2_pg = [1.5, 2.0, 2.5, 3.0]   # candidate revenue targets
_s2_search = {
    'profit_goal':  _s2_pg,
    'alpha_weight': [1e4, 1e5, 1e6],
    'beta_weight':  [5e2, 1e3, 5e3],
}
_s2_fixed = {
    'lr_theta':        lr_theta_star,
    'lr_w':            lr_w_star,
    'num_inner_steps': 200,
}
_s2_combos = [dict(zip(_s2_search, c))
              for c in itertools.product(*_s2_search.values())]
random.seed(SEED + 1); random.shuffle(_s2_combos)
print(f'Stage 2: {len(_s2_combos)} combos x {HP_S2_EPOCHS} epochs')

_s2_results = _run_stage(
    'S2-Weights', _s2_combos, _s2_fixed, HP_S2_EPOCHS, HP_SAMPLES,
    os.path.join(hp_save_dir, 's2_weights.json'), _search_start, _budget_s,
)

profit_goal_star  = _s2_results[0]['config']['profit_goal']
alpha_weight_star = _s2_results[0]['config']['alpha_weight']
beta_weight_star  = _s2_results[0]['config']['beta_weight']
print(f'\nStage 2 best:')
print(f'  profit_goal={profit_goal_star}  '
      f'alpha_weight={alpha_weight_star:.0e}  beta_weight={beta_weight_star:.0e}')
print(f'  score={_s2_results[0]["score"]:.4f}  revenue={_s2_results[0]["revenue_mean"]:.4f}')

# Sensitivity plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for _ax, _key, _star, _log in zip(
    axes,
    ['profit_goal', 'alpha_weight', 'beta_weight'],
    [profit_goal_star, alpha_weight_star, beta_weight_star],
    [False, True, True],
):
    _vals  = sorted(set(r['config'][_key] for r in _s2_results))
    _means = []
    for _v in _vals:
        _sub = [r['score'] for r in _s2_results if r['config'][_key] == _v]
        _ax.scatter([_v]*len(_sub), _sub, alpha=0.6, s=40)
        _means.append(sum(_sub)/len(_sub))
    _ax.plot(_vals, _means, 'k--', alpha=0.4, label='mean')
    _ax.axvline(_star, color='red', linestyle='--', label=f'best={_star}')
    if _log: _ax.set_xscale('log')
    _ax.set_xlabel(_key); _ax.set_ylabel('Score')
    _ax.legend(); _ax.set_title(f'Stage 2 — {_key}')
plt.tight_layout(); plt.show()


### Stage 3 — Fine-Tune Inner Oracle Steps

With the mechanism target locked in (from Stage 2), the difficulty of the IC problem is fixed. Too few inner steps and the oracle underestimates regret, producing a falsely optimistic IC loss; too many wastes compute with diminishing returns. We identify the knee of the IC-quality vs compute curve.

**Fixed:** all Stage 1 + 2 best values  
**Search:** `num_inner_steps` ∈ {100, 200, 300, 500} (**4 trials**)

In [ ]:
_s3_search = {'num_inner_steps': [100, 200, 300, 500]}
_s3_fixed = {
    'lr_theta':     lr_theta_star,
    'lr_w':         lr_w_star,
    'profit_goal':  profit_goal_star,
    'alpha_weight': alpha_weight_star,
    'beta_weight':  beta_weight_star,
}
_s3_combos = [{'num_inner_steps': n} for n in _s3_search['num_inner_steps']]
print(f'Stage 3: {len(_s3_combos)} combos x {HP_S3_EPOCHS} epochs')

_s3_results = _run_stage(
    'S3-InnerSteps', _s3_combos, _s3_fixed, HP_S3_EPOCHS, HP_SAMPLES,
    os.path.join(hp_save_dir, 's3_inner.json'), _search_start, _budget_s,
)

inner_steps_star = _s3_results[0]['config']['num_inner_steps']
print(f'\nStage 3 best: num_inner_steps={inner_steps_star}')
print(f'  score={_s3_results[0]["score"]:.4f}  revenue={_s3_results[0]["revenue_mean"]:.4f}')

# IC regret vs inner steps (the key diagnostic)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
_ns = [r['config']['num_inner_steps']      for r in _s3_results]
_sc = [r['score']                           for r in _s3_results]
_rg = [r.get('regret_mean', float('nan'))  for r in _s3_results]
axes[0].plot(_ns, _sc, 'o-b'); axes[0].set_xlabel('num_inner_steps')
axes[0].set_ylabel('Score');   axes[0].set_title('Stage 3 — Score')
axes[0].axvline(inner_steps_star, color='red', linestyle='--')
axes[1].plot(_ns, _rg, 's-', color='darkorange')
axes[1].set_xlabel('num_inner_steps')
axes[1].set_ylabel('IC Regret (eval)'); axes[1].set_title('IC Regret vs inner steps')
axes[1].axhline(HP_IC_LOSS_THRESH, color='red', linestyle=':', label=f'thresh={HP_IC_LOSS_THRESH}')
axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# ---- Consolidate optimal configuration from all 3 stages ----
best_cfg_hp = {
    'lr_theta':        lr_theta_star,
    'lr_w':            lr_w_star,
    'profit_goal':     profit_goal_star,
    'alpha_weight':    alpha_weight_star,
    'beta_weight':     beta_weight_star,
    'num_inner_steps': inner_steps_star,
}

print('=' * 62)
print('OPTIMAL HYPERPARAMETERS — Exp 2')
print('=' * 62)
for _k, _v in best_cfg_hp.items():
    print(f'  {_k:20s}: {_v}')
_tot = len(_s1_results) + len(_s2_results) + len(_s3_results)
_th  = (time.time() - _search_start) / 3600
print(f'\nTotal: {_tot} trials in {_th:.2f}h')

_search_summary = {
    'optimal_config':  best_cfg_hp,
    'stage1_best':     _s1_results[0] if _s1_results else None,
    'stage2_best':     _s2_results[0] if _s2_results else None,
    'stage3_best':     _s3_results[0] if _s3_results else None,
    'total_trials':    _tot,
    'total_time_h':    _th,
    'thresholds':      {'IC': HP_IC_LOSS_THRESH, 'IR': HP_IR_LOSS_THRESH,
                        'IR_viol': HP_IR_VIOL_THRESH, 'BV': HP_BV_THRESH},
}
with open(os.path.join(hp_save_dir, 'best_hparams.json'), 'w') as _f:
    json.dump(_search_summary, _f, indent=2)
print(f'Saved: {hp_save_dir}/best_hparams.json')


## Final Training with Optimal Hyperparameters

Retrain for the full epoch budget using the staged-search optimum. Note that `profit_goal` is now set to the value found in Stage 2, not the configuration default.

In [ ]:
print(f'Retraining — {cfg["num_outer_epochs"]} epochs with optimal config:')
for _k, _v in best_cfg_hp.items():
    print(f'  {_k:20s}: {_v}')

final_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
final_pnet = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=cfg['policy_hidden'],
    residual_init_std=cfg['residual_init_std'],
)
final_logs = nested_GD_budget(
    auction=auction,
    mechanism_network=final_mnet, policy_network=final_pnet,
    profit_goal=best_cfg_hp['profit_goal'],
    num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=best_cfg_hp['lr_theta'],
    num_inner_steps=best_cfg_hp['num_inner_steps'],
    lr_w=best_cfg_hp['lr_w'],
    num_samples_train=cfg['num_samples_train'],
    alpha_weight=best_cfg_hp['alpha_weight'],
    beta_weight=best_cfg_hp['beta_weight'],
    print_iter=cfg['print_iter'],
    seed=SEED,
)
final_mechanism_params = final_logs.pop('mechanism_params')
final_policy_params    = final_logs.pop('policy_params')
print('Final training complete.')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(final_logs['reg_plus']); axes[0].set_title('Regret+');    axes[0].set_xlabel('Epoch')
axes[1].plot(final_logs['gap_plus']); axes[1].set_title('Profit Gap+'); axes[1].set_xlabel('Epoch')
axes[2].plot(final_logs['ir']);        axes[2].set_title('IR Loss');     axes[2].set_xlabel('Epoch')
axes[3].plot(final_logs['loss']);      axes[3].set_title('Total Loss');  axes[3].set_xlabel('Epoch')
plt.suptitle(f'Exp 2 — Variant {VARIANT} (optimal hparams)', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
final_eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
final_results = eval_budget_mechanism_multikey(
    auction=auction, mechanism_network=final_mnet,
    mechanism_params=final_mechanism_params,
    profit_goal=best_cfg_hp['profit_goal'],
    num_samples=256, keys=final_eval_keys,
    policy_network=final_pnet, policy_params=final_policy_params,
)
print_eval_report(final_results, prefix=f'Exp 2 Variant {VARIANT} (optimal): ')


## Cross-Variant Comparison

Train A, B, C under identical optimal hyperparameters (including the `profit_goal` found in Stage 2). This is the core experimental comparison.

In [ ]:
variant_results = {
    VARIANT: {'logs': final_logs, 'eval': final_results,
              'mech_params': final_mechanism_params,
              'pol_params':  final_policy_params}
}
for _v in ['A', 'B', 'C']:
    if _v == VARIANT:
        continue
    print(f'\n--- Variant {_v} ---')
    _vnet  = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[_v]))
    _vpnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=cfg['policy_hidden'],
        residual_init_std=cfg['residual_init_std'],
    )
    _vl = nested_GD_budget(
        auction=auction,
        mechanism_network=_vnet, policy_network=_vpnet,
        profit_goal=best_cfg_hp['profit_goal'],
        num_outer_epochs=cfg['num_outer_epochs'],
        lr_theta=best_cfg_hp['lr_theta'],
        num_inner_steps=best_cfg_hp['num_inner_steps'],
        lr_w=best_cfg_hp['lr_w'],
        num_samples_train=cfg['num_samples_train'],
        alpha_weight=best_cfg_hp['alpha_weight'],
        beta_weight=best_cfg_hp['beta_weight'],
        print_iter=cfg['print_iter'],
        seed=SEED,
    )
    _vmp = _vl.pop('mechanism_params'); _vpp = _vl.pop('policy_params')
    _vek = jax.random.split(jax.random.PRNGKey(88888), 32)
    _vev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_vnet, mechanism_params=_vmp,
        profit_goal=best_cfg_hp['profit_goal'],
        num_samples=256, keys=_vek,
        policy_network=_vpnet, policy_params=_vpp,
    )
    print_eval_report(_vev, prefix=f'Variant {_v}: ')
    variant_results[_v] = {'logs': _vl, 'eval': _vev,
                            'mech_params': _vmp, 'pol_params': _vpp}

# Summary table
print('\n' + '='*72)
print(f'Exp 2 — A vs B vs C  (profit_goal={profit_goal_star}, optimal hparams)')
print('='*72)
print(f'  {"Var":>4} | {"Revenue":>9} | {"Gap+":>7} | {"IC Reg":>8} | {"IR Loss":>8} | {"BV":>6} | {"IV":>6}')
print('  ' + '-'*68)
for _v in ['A', 'B', 'C']:
    if _v not in variant_results: continue
    _ev = variant_results[_v]['eval']
    _rg = _ev.get('regret_stats', {}).get('mean', float('nan'))
    print(f'  {_v:>4} | {_ev["profit_stats"]["mean"]:>9.4f} | '
          f'{_ev["gap_plus_stats"]["mean"]:>7.4f} | '
          f'{_rg:>8.4f} | '
          f'{_ev["ir_loss_stats"]["mean"]:>8.4f} | '
          f'{_ev["budget_viol_stats"]["mean"]:>6.4f} | '
          f'{_ev["ir_viol_stats"]["mean"]:>6.4f}')

# Learning-curve comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for _v, _col in zip(['A','B','C'], ['tab:blue','tab:orange','tab:green']):
    if _v not in variant_results: continue
    _l = variant_results[_v]['logs']
    axes[0].plot(_l['reg_plus'], color=_col, label=_v, alpha=0.85)
    axes[1].plot(_l['gap_plus'], color=_col, label=_v, alpha=0.85)
    axes[2].plot(_l['ir'],       color=_col, label=_v, alpha=0.85)
for _ax, _t in zip(axes, ['Regret+','Profit Gap+','IR Loss']):
    _ax.set_title(_t); _ax.set_xlabel('Epoch'); _ax.legend()
plt.suptitle(f'Exp 2 — A vs B vs C (optimal hparams)', y=1.02)
plt.tight_layout(); plt.show()


## Save All Outputs

In [ ]:
final_save_dir = f'outputs/private_budget_multigood/seed{SEED}/optimal'
os.makedirs(final_save_dir, exist_ok=True)

variant_summary = {}
for _v, _vr in variant_results.items():
    with open(f'{final_save_dir}/mech_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['mech_params'], _f)
    with open(f'{final_save_dir}/pol_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['pol_params'], _f)
    with open(f'{final_save_dir}/logs_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['logs'], _f)
    _ev = _vr['eval']
    variant_summary[_v] = {
        'revenue':        _ev['profit_stats'],
        'gap_plus':       _ev['gap_plus_stats'],
        'ir_loss':        _ev['ir_loss_stats'],
        'budget_viol':    _ev['budget_viol_stats'],
        'ir_viol':        _ev['ir_viol_stats'],
        'regret':         _ev.get('regret_stats', {}),
        'optimal_config': best_cfg_hp,
    }
with open(f'{final_save_dir}/variant_comparison.json', 'w') as _f:
    json.dump(variant_summary, _f, indent=2)
with open(f'{final_save_dir}/optimal_hparams.json', 'w') as _f:
    json.dump({'optimal_config': best_cfg_hp,
               'exp': 'Exp 2', 'variant': VARIANT}, _f, indent=2)

print(f'Saved to: {final_save_dir}/')
print(f'  mech/pol/logs_{{A,B,C}}.pkl — model artefacts per variant')
print(f'  variant_comparison.json   — cross-variant eval summary')
print(f'  optimal_hparams.json      — best hyperparameter config')
print(f'HP search logs: {hp_save_dir}/')
print(f'  s1_lr.json / s2_weights.json / s3_inner.json — per-stage results')
print(f'  best_hparams.json — consolidated search summary')
